In [ ]:
# ==========================================
# 1. COLAB SETUP: DRIVE + REPO CODE
# ==========================================
import os
import pathlib
import subprocess
import sys

from google.colab import drive
from google.colab import userdata

REPO_URL = "https://github.com/DrHBSB/RadLE_CRASH_Lab.git"
REPO_DIR = pathlib.Path("/content/RadLE_CRASH_Lab")
SRC_DIR = REPO_DIR / "src"

# Drive remains the home for images and benchmark outputs.
drive.mount("/content/drive")

# Colab does not automatically check out the full GitHub repo when opening a notebook.
# For this private repo, add a Colab secret named GITHUB_TOKEN if clone needs auth.
if not (SRC_DIR / "radle_benchmark.py").exists():
    try:
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

    clone_url = REPO_URL
    if github_token:
        clone_url = REPO_URL.replace("https://", f"https://{github_token}@")
    subprocess.run(["git", "clone", clone_url, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
# ==========================================
# 2. API CLIENT + BENCHMARK IMPORTS
# ==========================================
from openai import OpenAI

from radle_benchmark import create_scorer_view, run_benchmark

try:
    os.environ["TEST_OPENROUTER_API_KEY"] = userdata.get("TEST_OPENROUTER_API_KEY")
except Exception:
    print("ERROR: Could not find TEST_OPENROUTER_API_KEY in Colab Secrets.")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["TEST_OPENROUTER_API_KEY"],
)


In [ ]:
# ==========================================
# 3. DRIVE PATHS + RUN CONFIG
# ==========================================
master_images_folder = "/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/RadLE v2 Master Data"
final_output_csv = "/content/drive/MyDrive/CRASH Lab/RaDLE/CONFIDENTIAL/RadLE v2 Dataset/RadLE_v1.5_RSNA.csv"

# Recommended sequence: run TEST_LIMIT=1 first; if syntax/API plumbing passes, run TEST_LIMIT=5.
# Use TEST_LIMIT=None only when ready for the full benchmark.
TEST_LIMIT = 1


In [ ]:
# ==========================================
# 4. RUN BENCHMARK
# ==========================================
df_final = run_benchmark(
    client=client,
    image_folder=master_images_folder,
    output_csv=final_output_csv,
    test_limit=TEST_LIMIT,
)

print("\nFINAL DATAFRAME PREVIEW:")
from IPython.display import display

display(df_final.head())


In [ ]:
# ==========================================
# 5. SCORER'S VIEW & PIVOT
# ==========================================
df_scorer, display_df, scorer_csv = create_scorer_view(final_output_csv)

print(f"Scorer version saved to: {scorer_csv}")
print("\nTRANSPOSED CONSENSUS VIEW:")
display(display_df)
